<div align='center'>
    <h1>Implementing a Fuzzy Logic System to Evaluate Video Game Performance Based on Metadata and Platform Distribution Featur</h1>
    <h3>DKA PROJECT by STEPMTOHER LOVER</h3>
</div>

In [4]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 

In [5]:
data_games = pd.read_csv('C:\\tugas\\TUBES DKA\\Dataset\\games_metadata_5k.csv')
data_ratings = pd.read_csv('C:\\tugas\\TUBES DKA\\Dataset\\game_ratings.csv')

In [6]:
#Fuzzyfication
def trimf(x, a, b, c): 
    left = (x - a) / (b - a) if b != a else np.zeros_like(x)
    right = (x - c) / (b - c) if b != c else np.zeros_like(x)
    return np.maximum(0, np.minimum(left, right))

In [7]:
#Define the universe 
x_global = np.linspace(0, 5, 500)
x_avg_user_rating = np.linspace(1, 5, 500)
x_rating_count =np.linspace(0, 100, 500)
x_platform = np.linspace(1, 22, 500)
x_recent = np.linspace(0, 10, 500)
x_recommendation = np.linspace(0, 10, 500)

In [10]:
#Membership function 
mf_global = {
    'Poor' : trimf(x_global, 0, 0, 2.5),
    'Average' : trimf(x_global, 1.5, 2.5, 3.5),
    'High' : trimf(x_global, 3, 5, 5),
}
mf_avg_rating = {
    'Low' : trimf(x_avg_user_rating,1, 1, 3), 
    'Medium' : trimf(x_avg_user_rating,2, 3, 4), 
    'High' : trimf(x_avg_user_rating, 3, 5, 5),
}
mf_count = {
    'Few' : trimf(x_rating_count, 0, 0, 40),
    'Moderate' : trimf(x_rating_count, 20, 50, 80), 
    'Many' : trimf(x_rating_count, 60, 100, 100),
}

mf_platform = {
    'Narrow' : trimf(x_platform, 1, 1, 6), 
    'Medium' : trimf(x_platform, 3, 7, 12), 
    'Wide' : trimf(x_platform, 8, 22, 22),
}

mf_recent = {
    'Old' : trimf(x_recent, 0, 0, 4), 
    'Recent' : trimf(x_recent, 3, 5, 7), 
    'New' : trimf(x_recent, 6, 10, 10), 
}

mf_recommendation = {
    'Poor' : trimf(x_recommendation, 0, 0, 4), 
    'Fair' : trimf(x_recommendation, 3, 5, 7),
    'Excellent' : trimf(x_recommendation, 6, 10, 10),
    
}

In [12]:
#Rules and stuff
def fuzzy_and(*args): 
    return min(args)
def fuzzy_or(*args): 
    return max(args)

In [ ]:
#manual trimf helper that replicates skfuzzy's bitch ass
import numpy as np

def trimf(x, abc):
    """
    Replicates skfuzzy.trimf mathematically.
    abc is a list/tuple of three points [a, b, c].
    """
    a, b, c = abc
    
    # Handle edge cases for flat/half-triangles at boundaries
    if a == b:
        # Downward slope from b to c
        return np.where(x < b, 0.0, np.where(x <= c, (c - x) / (c - b), 0.0))
    if b == c:
        # Upward slope from a to b
        return np.where(x < a, 0.0, np.where(x <= b, (x - a) / (b - a), 0.0))
        
    # Standard triangle
    first_half = (x - a) / (b - a)
    second_half = (c - x) / (c - b)
    return np.maximum(0, np.minimum(first_half, second_half))


In [ ]:
#mayor mamdani

def mamdani_inference(global_in, avg_rating_in, count_in, platform_in, recent_in):
    # --- 1. FUZZIFICATION ---
    # Global
    g_poor    = trimf(global_in, [0, 0, 2.5])
    g_average = trimf(global_in, [1.5, 2.5, 3.5])
    g_high    = trimf(global_in, [3, 5, 5])
    
    # Average User Rating
    r_low    = trimf(avg_rating_in, [1, 1, 3])
    r_medium = trimf(avg_rating_in, [2, 3, 4])
    r_high   = trimf(avg_rating_in, [3, 5, 5])
    
    # Rating Count
    c_few      = trimf(count_in, [0, 0, 40])
    c_moderate = trimf(count_in, [20, 50, 80])
    c_many     = trimf(count_in, [60, 100, 100])
    
    # Platform
    p_narrow = trimf(platform_in, [1, 1, 6])
    p_medium = trimf(platform_in, [3, 7, 12])
    p_wide   = trimf(platform_in, [8, 22, 22])
    
    # Recent
    rec_old    = trimf(recent_in, [0, 0, 4])
    rec_recent = trimf(recent_in, [3, 5, 7])
    rec_new    = trimf(recent_in, [6, 10, 10])

    # --- 2. RULE EVALUATION & IMPLICATION ---
    # Discretize the output universe (Recommendation from 0 to 10)
    x_recommendation = np.linspace(0, 10, 500)
    
    # Define the output membership shapes
    rec_poor      = trimf(x_recommendation, [0, 0, 4])
    rec_fair      = trimf(x_recommendation, [3, 5, 7])
    rec_excellent = trimf(x_recommendation, [6, 10, 10])
    
    # Example Rules (You can modify these configurations to match your logic):
    # Rule 1: IF global is Poor OR avg_rating is Low THEN recommendation is Poor
    w1 = np.maximum(g_poor, r_low)
    rule1_clipped = np.minimum(w1, rec_poor)
    
    # Rule 2: IF global is Average AND count is Moderate THEN recommendation is Fair
    w2 = np.minimum(g_average, c_moderate)
    rule2_clipped = np.minimum(w2, rec_fair)
    
    # Rule 3: IF global is High AND recent is New AND platform is Wide THEN recommendation is Excellent
    w3 = np.minimum(np.minimum(g_high, rec_new), p_wide)
    rule3_clipped = np.minimum(w3, rec_excellent)

    # --- 3. AGGREGATION ---
    aggregated = np.maximum(rule1_clipped, np.maximum(rule2_clipped, rule3_clipped))
    
    # --- 4. DEFUZZIFICATION (Centroid Method) ---
    numerator = np.sum(x_recommendation * aggregated)
    denominator = np.sum(aggregated)
    
    if denominator == 0:
        return 5.0 # Return the midpoint if no rules fire
    return numerator / denominator

# Example Test Run
mamdani_output = mamdani_inference(global_in=4.5, avg_rating_in=4.8, count_in=85, platform_in=18, recent_in=9)
print(f"Mamdani Recommendation Score: {mamdani_output:.2f} / 10")


In [ ]:
#sugenoooooooooo

def sugeno_inference(global_in, avg_rating_in, count_in, platform_in, recent_in):
    # --- 1. FUZZIFICATION ---
    g_poor    = trimf(global_in, [0, 0, 2.5])
    g_average = trimf(global_in, [1.5, 2.5, 3.5])
    g_high    = trimf(global_in, [3, 5, 5])
    
    r_low    = trimf(avg_rating_in, [1, 1, 3])
    r_medium = trimf(avg_rating_in, [2, 3, 4])
    r_high   = trimf(avg_rating_in, [3, 5, 5])
    
    c_few      = trimf(count_in, [0, 0, 40])
    c_moderate = trimf(count_in, [20, 50, 80])
    c_many     = trimf(count_in, [60, 100, 100])
    
    p_narrow = trimf(platform_in, [1, 1, 6])
    p_medium = trimf(platform_in, [3, 7, 12])
    p_wide   = trimf(platform_in, [8, 22, 22])
    
    rec_old    = trimf(recent_in, [0, 0, 4])
    rec_recent = trimf(recent_in, [3, 5, 7])
    rec_new    = trimf(recent_in, [6, 10, 10])

    # --- 2. SUGENO CONSTANT CONSEQUENTS ---
    # Based on the peaks of your image's recommendation metrics
    OUT_POOR = 0.0
    OUT_FAIR = 5.0
    OUT_EXCELLENT = 10.0

    # --- 3. RULE EVALUATION (Firing Strengths) ---
    # Rule 1: IF global is Poor OR avg_rating is Low THEN Out = Poor
    w1 = np.maximum(g_poor, r_low)
    
    # Rule 2: IF global is Average AND count is Moderate THEN Out = Fair
    w2 = np.minimum(g_average, c_moderate)
    
    # Rule 3: IF global is High AND recent is New AND platform is Wide THEN Out = Excellent
    w3 = np.minimum(np.minimum(g_high, rec_new), p_wide)
    
    rules = [
        {'weight': w1, 'output': OUT_POOR},
        {'weight': w2, 'output': OUT_FAIR},
        {'weight': w3, 'output': OUT_EXCELLENT}
    ]

    # --- 4. DEFUZZIFICATION (Weighted Average) ---
    numerator = sum(r['weight'] * r['output'] for r in rules)
    denominator = sum(r['weight'] for r in rules)
    
    if denominator == 0:
        return 5.0 # Return the midpoint if no rules fire
    return numerator / denominator

# Example Test Run
sugeno_output = sugeno_inference(global_in=4.5, avg_rating_in=4.8, count_in=85, platform_in=18, recent_in=9)
print(f"Sugeno Recommendation Score:  {sugeno_output:.2f} / 10")


In [ ]:
print("Farrell Was Here")